# 03 - TF-IDF Keyword Extraction

This notebook uses TF-IDF (Term Frequency-Inverse Document Frequency) to:
- Extract the most important keywords from CVE descriptions
- Identify common vulnerability terms and attack patterns
- Visualize the most significant terms
- Build a TF-IDF feature matrix for downstream models

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from sklearn.feature_extraction.text import TfidfVectorizer
from collections import Counter
import joblib
import sys
sys.path.append('..')

plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style('whitegrid')

## 1. Load Preprocessed Data

In [ ]:
df = pd.read_csv('../data/cve_preprocessed.csv')
print(f"Loaded {len(df)} CVE records")

# Use cleaned descriptions for TF-IDF
texts = df['Cleaned_Description'].fillna('').tolist()
print(f"Non-empty descriptions: {sum(1 for t in texts if t.strip())}")

## 2. Build TF-IDF Matrix

In [ ]:
# Configure TF-IDF Vectorizer
tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,      # Limit features for performance
    min_df=2,               # Must appear in at least 2 documents
    max_df=0.95,            # Ignore terms in >95% of documents
    ngram_range=(1, 2),     # Unigrams and bigrams
    sublinear_tf=True       # Apply sublinear TF scaling
)

tfidf_matrix = tfidf_vectorizer.fit_transform(texts)
feature_names = tfidf_vectorizer.get_feature_names_out()

print(f"TF-IDF Matrix shape: {tfidf_matrix.shape}")
print(f"  Documents: {tfidf_matrix.shape[0]}")
print(f"  Features (terms): {tfidf_matrix.shape[1]}")
print(f"  Sparsity: {1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1]):.4f}")

## 3. Top Global TF-IDF Terms

In [ ]:
# Calculate mean TF-IDF score across all documents
mean_tfidf = np.array(tfidf_matrix.mean(axis=0)).flatten()
top_indices = mean_tfidf.argsort()[::-1][:30]

top_terms = [(feature_names[i], mean_tfidf[i]) for i in top_indices]

print("Top 30 Terms by Mean TF-IDF Score:")
print("=" * 50)
for term, score in top_terms:
    print(f"  {term:30s} {score:.4f}")

# Visualization
fig = px.bar(x=[t[1] for t in top_terms], y=[t[0] for t in top_terms],
             orientation='h',
             labels={'x': 'Mean TF-IDF Score', 'y': 'Term'},
             title='Top 30 Terms by Mean TF-IDF Score Across All CVEs',
             color=[t[1] for t in top_terms],
             color_continuous_scale='Reds')
fig.update_layout(yaxis={'categoryorder': 'total ascending'}, height=700, showlegend=False)
fig.show()

## 4. Extract Top Keywords per CVE

In [ ]:
def get_top_keywords(tfidf_row, feature_names, n=5):
    """Extract top-n keywords from a single document's TF-IDF vector."""
    row_array = tfidf_row.toarray().flatten()
    top_indices = row_array.argsort()[::-1][:n]
    return [(feature_names[i], row_array[i]) for i in top_indices if row_array[i] > 0]

# Extract keywords for first 10 CVEs
print("Top Keywords per CVE (sample):")
print("=" * 80)
for i in range(min(10, len(df))):
    keywords = get_top_keywords(tfidf_matrix[i], feature_names, n=5)
    kw_str = ', '.join([f"{kw[0]} ({kw[1]:.3f})" for kw in keywords])
    print(f"\n{df['CVE ID'].iloc[i]}:")
    print(f"  Keywords: {kw_str}")

In [ ]:
# Add top keywords column to dataframe
all_keywords = []
for i in range(len(df)):
    keywords = get_top_keywords(tfidf_matrix[i], feature_names, n=5)
    kw_list = [kw[0] for kw in keywords]
    all_keywords.append(', '.join(kw_list))

df['Top_Keywords'] = all_keywords
df[['CVE ID', 'Top_Keywords']].head(10)

## 5. Keywords by Vulnerability Type

In [ ]:
# Analyze top terms per vulnerability type
vuln_types = df['Vulnerability_Type'].unique()
vuln_types = [v for v in vuln_types if v != 'Other']

print("Top Keywords by Vulnerability Type:")
print("=" * 60)

for vtype in sorted(vuln_types):
    mask = df['Vulnerability_Type'] == vtype
    if mask.sum() < 2:
        continue
    type_tfidf = tfidf_matrix[mask]
    type_mean = np.array(type_tfidf.mean(axis=0)).flatten()
    top_idx = type_mean.argsort()[::-1][:8]
    top_terms_type = [feature_names[i] for i in top_idx if type_mean[i] > 0]
    print(f"\n{vtype}:")
    print(f"  {', '.join(top_terms_type)}")

## 6. Keywords by Severity Level

In [ ]:
# Top keywords by severity
for severity in ['Critical', 'High', 'Medium', 'Low']:
    mask = df['Severity'] == severity
    if mask.sum() < 2:
        continue
    sev_tfidf = tfidf_matrix[mask]
    sev_mean = np.array(sev_tfidf.mean(axis=0)).flatten()
    top_idx = sev_mean.argsort()[::-1][:10]
    top_terms_sev = [(feature_names[i], sev_mean[i]) for i in top_idx]
    print(f"\n{severity} Severity - Top Terms:")
    for term, score in top_terms_sev:
        print(f"  {term:25s} {score:.4f}")

## 7. Security-Specific Pattern Detection

In [ ]:
# Search for specific attack pattern terms in the TF-IDF vocabulary
attack_patterns = [
    'remote execution', 'buffer overflow', 'sql injection',
    'denial service', 'cross site', 'privilege escalation',
    'code execution', 'command injection', 'path traversal',
    'information disclosure', 'authentication bypass'
]

print("Attack Pattern TF-IDF Presence:")
print("=" * 60)
feature_set = set(feature_names)
for pattern in attack_patterns:
    if pattern in feature_set:
        idx = list(feature_names).index(pattern)
        doc_freq = (tfidf_matrix[:, idx].toarray() > 0).sum()
        avg_score = tfidf_matrix[:, idx].toarray().mean()
        print(f"  '{pattern}': in {doc_freq} documents, avg TF-IDF: {avg_score:.4f}")
    else:
        print(f"  '{pattern}': not in vocabulary (may appear differently)")

## 8. Save TF-IDF Artifacts

In [ ]:
import scipy.sparse

# Save TF-IDF vectorizer
joblib.dump(tfidf_vectorizer, '../models/tfidf_vectorizer.joblib')

# Save TF-IDF matrix
scipy.sparse.save_npz('../models/tfidf_matrix.npz', tfidf_matrix)

# Save updated dataframe
df.to_csv('../data/cve_with_keywords.csv', index=False)

print("Saved:")
print("  - TF-IDF vectorizer: models/tfidf_vectorizer.joblib")
print("  - TF-IDF matrix: models/tfidf_matrix.npz")
print("  - Updated dataset: data/cve_with_keywords.csv")
print("\n✅ TF-IDF keyword extraction complete!")